# Crypto Wallet Allocation Optimizer: Hot vs Cold Strategy


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Crypto Wallet Allocation Optimizer: Hot vs Cold Strategy

This notebook helps traders decide how to split cryptocurrency between hot (online, liquid) and cold (offline, secure) wallets based on trading activity, risk profile, and portfolio size.

**Source:** Analysis based on wallet security trade-off principles from 'How to Choose Between Hot and Cold Wallets: A Trader's Guide'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from enum import Enum

class RiskProfile(Enum):
    CONSERVATIVE = 0.1
    MODERATE = 0.3
    AGGRESSIVE = 0.5

class TradingFrequency(Enum):
    LOW = 1
    MEDIUM = 3
    HIGH = 5

def calculate_wallet_allocation(portfolio_value_usd, trading_frequency, risk_profile, hold_duration_days=30):
    """
    Calculate recommended hot/cold wallet split.
    
    Args:
        portfolio_value_usd: Total portfolio value in USD
        trading_frequency: TradingFrequency enum (trades per week)
        risk_profile: RiskProfile enum (fraction for hot wallet)
        hold_duration_days: Average days holding positions
    
    Returns:
        dict with hot_allocation_pct, cold_allocation_pct, and reasoning
    """
    base_hot_allocation = risk_profile.value
    freq_multiplier = trading_frequency.value / 5.0
    hold_adjustment = max(0.05, 1.0 - (hold_duration_days / 90.0))
    
    hot_allocation = min(0.7, base_hot_allocation * freq_multiplier * hold_adjustment)
    cold_allocation = 1.0 - hot_allocation
    
    return {
        'hot_allocation_pct': round(hot_allocation * 100, 1),
        'cold_allocation_pct': round(cold_allocation * 100, 1),
        'hot_amount_usd': round(portfolio_value_usd * hot_allocation, 2),
        'cold_amount_usd': round(portfolio_value_usd * cold_allocation, 2),
        'risk_level': risk_profile.name,
        'trading_frequency': trading_frequency.name
    }

print("Wallet Allocation Calculator Initialized")

## Input Your Trading Profile

Specify your portfolio size, trading activity, and risk tolerance to get a personalized allocation recommendation.

In [ ]:
portfolio_value = 50000
trading_freq = TradingFrequency.MEDIUM
risk = RiskProfile.MODERATE
avg_hold_days = 14

allocation = calculate_wallet_allocation(
    portfolio_value_usd=portfolio_value,
    trading_frequency=trading_freq,
    risk_profile=risk,
    hold_duration_days=avg_hold_days
)

print(f"\nRecommended Allocation for ${portfolio_value:,} Portfolio:")
print(f"  Hot Wallet:  {allocation['hot_allocation_pct']}% (${allocation['hot_amount_usd']:,.2f})")
print(f"  Cold Wallet: {allocation['cold_allocation_pct']}% (${allocation['cold_amount_usd']:,.2f})")
print(f"\n  Profile: {allocation['risk_level']} Risk, {allocation['trading_frequency']} Frequency")

## Scenario Analysis: Compare Different Strategies

In [ ]:
scenarios = []
for freq in [TradingFrequency.LOW, TradingFrequency.MEDIUM, TradingFrequency.HIGH]:
    for risk in [RiskProfile.CONSERVATIVE, RiskProfile.MODERATE, RiskProfile.AGGRESSIVE]:
        alloc = calculate_wallet_allocation(
            portfolio_value_usd=100000,
            trading_frequency=freq,
            risk_profile=risk,
            hold_duration_days=14
        )
        scenarios.append({
            'Trading Frequency': freq.name,
            'Risk Profile': risk.name,
            'Hot %': alloc['hot_allocation_pct'],
            'Cold %': alloc['cold_allocation_pct'],
            'Hot ($)': alloc['hot_amount_usd'],
            'Cold ($)': alloc['cold_amount_usd']
        })

df_scenarios = pd.DataFrame(scenarios)
print("\nAllocation Matrix for $100,000 Portfolio (14-day avg hold):")
print(df_scenarios.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

freq_data = df_scenarios[df_scenarios['Risk Profile'] == 'MODERATE']
axes[0].barh(freq_data['Trading Frequency'], freq_data['Hot %'], label='Hot', color='#FF6B6B')
axes[0].barh(freq_data['Trading Frequency'], freq_data['Cold %'], left=freq_data['Hot %'], label='Cold', color='#4ECDC4')
axes[0].set_xlabel('Allocation %')
axes[0].set_title('Hot vs Cold by Trading Frequency\n(Moderate Risk)')
axes[0].legend()
axes[0].set_xlim(0, 100)

risk_data = df_scenarios[df_scenarios['Trading Frequency'] == 'MEDIUM']
axes[1].barh(risk_data['Risk Profile'], risk_data['Hot %'], label='Hot', color='#FF6B6B')
axes[1].barh(risk_data['Risk Profile'], risk_data['Cold %'], left=risk_data['Hot %'], label='Cold', color='#4ECDC4')
axes[1].set_xlabel('Allocation %')
axes[1].set_title('Hot vs Cold by Risk Profile\n(Medium Trading Frequency)')
axes[1].legend()
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

print("\nVisualization complete.")

## Security and Liquidity Trade-off Analysis

Compute estimated access time and security risk by allocation strategy.

In [ ]:
def security_liquidity_analysis(hot_pct):
    """
    Estimate security risk and liquidity benefit by allocation.
    """
    cold_pct = 100 - hot_pct
    
    security_score = cold_pct
    avg_access_hours = (hot_pct / 100) * 0.25 + (cold_pct / 100) * 24
    breach_risk_score = hot_pct * 0.008
    
    return {
        'hot_allocation': hot_pct,
        'security_score_0_100': round(security_score, 1),
        'avg_access_time_hours': round(avg_access_hours, 2),
        'daily_breach_risk_bps': round(breach_risk_score, 2)
    }

print("\nSecurity & Liquidity Trade-off (per allocation strategy):")
print(f"{'Hot %':<8} {'Security':<12} {'Access Time':<15} {'Daily Risk (bps)':<15}")
print("-" * 50)
for hot_pct in [10, 20, 30, 40, 50, 60]:
    analysis = security_liquidity_analysis(hot_pct)
    print(f"{analysis['hot_allocation']:<8} {analysis['security_score_0_100']:<12} {analysis['avg_access_time_hours']:<15} {analysis['daily_breach_risk_bps']:<15}")

---

## Reference

[protraderdaily](https://protraderdaily.com/crypto/how-to-choose-between-hot-and-cold-wallets)
